# WP6 — Conformal sufficiency layer (Student 3, Milestone 3)

**What this notebook does.** Reads the input contract (probability_table, labels, covariates), applies split conformal prediction with Adaptive Prediction Sets under leave-one-subject-out cross-validation, and emits three artefacts per pipeline contract §5 + §7:
- `prediction_sets.parquet` — |C_α|, scores, stratum per (participant, night)
- `decisions/covariate_conditional.parquet` — predict/defer decision per night
- `tau_per_strategy.parquet` (appended) — per-participant τᵢ under this strategy

Also produces the sufficiency curves plot (Figure 1 Panel A draft) that is Student 3's Milestone-3 gate.

**Scope of this starter.**
- Global q̂ (no Mondrian yet) — Mondrian stratification is Milestone 4.
- APS specialised to binary classification.
- α = 0.10 (plan §2.3).
- Calibration: for each held-out participant, use all other participants' all (participant, night) predictions as calibration.


In [ ]:
import sys, os, subprocess
from pathlib import Path

REPO_ROOT = Path().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd
from utils.preview import peek, summary

try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
except ImportError:
    pass

DATA = Path('synthetic/v1')
DATA.mkdir(parents=True, exist_ok=True)
(DATA / 'preprocessing').mkdir(exist_ok=True)
(DATA / 'endpoint').mkdir(exist_ok=True)
(DATA / 'decisions').mkdir(exist_ok=True)
(DATA / 'evaluation').mkdir(exist_ok=True)
print(f'Pipeline root: {DATA}')


## 1. Load inputs


In [ ]:
prob = pd.read_parquet(DATA / 'probability_table.parquet')
lbls = pd.read_parquet(DATA / 'labels.parquet')
cov = pd.read_parquet(DATA / 'covariates.parquet')
joined = prob.merge(lbls, on=['participant_id', 'night_index'])
print(f'joined rows: {len(joined):,}')


## 2. Non-conformity scores (APS for binary)

For each (participant, night) with true label y and predicted p̂(post|x):
$$s(x,y) = 1 - \hat{p}(y\mid x).$$


In [ ]:
joined['p_true'] = np.where(joined['binary_label'] == 'post',
                             joined['p_post_ovulatory'],
                             1.0 - joined['p_post_ovulatory'])
joined['nonconformity'] = 1.0 - joined['p_true']
joined[['participant_id','night_index','binary_label','p_post_ovulatory','nonconformity']].head(8)


## 3. LOSO conformal quantile — per-k calibration (recipe b)

**Calibration recipe (awaiting supervisor sign-off; see pipeline contract §11).**
For each held-out participant i and test night k, the calibration set is the other 41 participants' nonconformity scores **at the same cumulative_k**. Rationale: the sufficiency framing is about uncertainty at a specific cumulative night; pooling scores across k (recipe a) mixes the early-uncertain regime with the late-confident regime and produces inverted trajectories. Recipe b gives a q̂ that varies with k, which is what lets |Cα| start at 2 and shrink to 1 as evidence accumulates.


In [ ]:
ALPHA = 0.10
pids = sorted(prob['participant_id'].unique())
# Pre-compute q̂ for every (pid, k) pair using scores from other participants at the same k.
q_hat_table = {}
for pid in pids:
    others = joined[joined['participant_id'] != pid]
    for k, sub in others.groupby('cumulative_k'):
        cal = sub['nonconformity'].to_numpy()
        n = len(cal)
        level = min(np.ceil((n + 1) * (1 - ALPHA)) / n, 1.0)
        q_hat_table[(pid, int(k))] = float(np.quantile(cal, level, method='higher'))
n_total = len(q_hat_table)
median_q = float(np.median(list(q_hat_table.values())))
print(f'computed q̂ for {n_total:,} (pid, k) pairs; median q̂ across table: {median_q:.3f}')


## 4. Construct prediction sets

For each held-out participant's nights: include class $y$ in $C_\alpha$ iff $\hat{p}(y\mid x) \geq 1 - \hat{q}_k$, where $\hat{q}_k$ depends on the night's cumulative-k.


In [ ]:
rows = []
for pid in pids:
    sub = joined[joined['participant_id'] == pid].sort_values('night_index')
    for _, r in sub.iterrows():
        k = int(r['cumulative_k'])
        q_hat = q_hat_table.get((pid, k))
        if q_hat is None:
            # participant i at cumulative_k that no other participant reached — skip or fallback
            continue
        p_post = r['p_post_ovulatory']; p_pre = 1.0 - p_post
        contains_post = p_post >= 1 - q_hat
        contains_pre = p_pre >= 1 - q_hat
        # Guarantee |C| ≥ 1 by including argmax if both would otherwise be excluded
        # (basic APS can produce empty sets; Romano et al. 2020 §3 discusses a
        # nonempty variant, but fallback-to-argmax is equivalent for the binary case).
        if not contains_pre and not contains_post:
            if p_post >= p_pre:
                contains_post = True
            else:
                contains_pre = True
        rows.append({
            'participant_id': pid,
            'night_index': int(r['night_index']),
            'cumulative_k': k,
            'alpha': ALPHA,
            'contains_pre': bool(contains_pre),
            'contains_post': bool(contains_post),
            'set_size': int(contains_pre) + int(contains_post),
            'nonconformity_score_pre': float(1 - p_pre),
            'nonconformity_score_post': float(1 - p_post),
            'conformal_quantile': float(q_hat),
            'mondrian_stratum': 'global',
            'is_synthetic': True,
        })
ps = pd.DataFrame(rows).astype({
    'participant_id': 'string', 'night_index': 'int32', 'cumulative_k': 'int32',
    'alpha': 'float32', 'contains_pre': 'bool', 'contains_post': 'bool',
    'set_size': 'int8', 'nonconformity_score_pre': 'float32',
    'nonconformity_score_post': 'float32', 'conformal_quantile': 'float32',
    'mondrian_stratum': 'string', 'is_synthetic': 'bool',
})
ps.to_parquet(DATA / 'prediction_sets.parquet', index=False)
print(f'Prediction sets written: {len(ps):,} rows. Set-size distribution:')
print(ps['set_size'].value_counts().sort_index())


## 5. Compute τᵢ and emit decision file

Decision rule (pipeline contract §5, covariate_conditional):
- `predict` at first k where `set_size == 1` for 2 consecutive nights.
- `defer` when `set_size == 2`.
- `collect_more` when `valid_nights.quality_flag` is `low_signal`/`dropout` (v1: not yet emitted because synthetic has no low-quality nights).


In [ ]:
dec_rows, tau_rows = [], []
for pid in pids:
    g = ps[ps['participant_id'] == pid].sort_values('night_index').reset_index(drop=True)
    tau = None
    for i in range(len(g) - 1):
        if g.loc[i, 'set_size'] == 1 and g.loc[i+1, 'set_size'] == 1:
            tau = int(g.loc[i, 'night_index'])
            break
    for i, r in g.iterrows():
        committed = tau is not None and r['night_index'] >= tau
        dec_rows.append({
            'participant_id': pid,
            'night_index': int(r['night_index']),
            'cumulative_k': int(r['cumulative_k']),
            'strategy_name': 'covariate_conditional',
            'decision': 'predict' if committed else 'defer',
            'predicted_label': ('post' if r['contains_post'] and not r['contains_pre'] else
                                'pre'  if r['contains_pre'] and not r['contains_post'] else None) if committed else None,
            'prediction_set_size': int(r['set_size']),
            'mondrian_stratum': 'global',
            'is_synthetic': True,
        })
    status = 'converged' if tau is not None else 'non_converger'
    tau_rows.append({'participant_id': pid, 'strategy_name': 'covariate_conditional',
                     'tau_i': tau, 'convergence_status': status, 'is_synthetic': True})

decisions = pd.DataFrame(dec_rows).astype({
    'participant_id': 'string', 'night_index': 'int32', 'cumulative_k': 'int32',
    'strategy_name': 'string', 'decision': 'string',
    'predicted_label': 'string', 'prediction_set_size': 'Int8',
    'mondrian_stratum': 'string', 'is_synthetic': 'bool',
})
decisions.to_parquet(DATA / 'decisions/covariate_conditional.parquet', index=False)
print(f'decisions/covariate_conditional.parquet: {len(decisions):,} rows, '
      f'{(decisions["decision"]=="predict").sum()} predict, '
      f'{(decisions["decision"]=="defer").sum()} defer')


## 6. Append τᵢ to `tau_per_strategy.parquet`


In [ ]:
tau_new = pd.DataFrame(tau_rows).astype({
    'participant_id': 'string', 'strategy_name': 'string',
    'tau_i': 'Int32', 'convergence_status': 'string', 'is_synthetic': 'bool'})

tau_path = DATA / 'tau_per_strategy.parquet'
if tau_path.exists():
    existing = pd.read_parquet(tau_path)
    existing = existing[existing['strategy_name'] != 'covariate_conditional']
    combined = pd.concat([existing, tau_new], ignore_index=True)
else:
    combined = tau_new
combined.to_parquet(tau_path, index=False)
print('tau_per_strategy.parquet strategy counts:')
print(combined.groupby('strategy_name').size())


## 7. Milestone-3 plot: |Cα| vs k for representative participants

This is the gate that Student 3 sends to supervisor at Milestone 3.


In [ ]:
import matplotlib.pyplot as plt
ss = cov.set_index('participant_id')['signal_completeness']
sample_pids = (ss.sort_values().iloc[[0, 7, 14, 21, 27, 34, 41]].index.tolist())
fig, ax = plt.subplots(figsize=(8, 4.5))
for pid in sample_pids:
    g = ps[ps['participant_id'] == pid].sort_values('night_index')
    ax.plot(g['night_index'], g['set_size'], alpha=0.8, label=f'{pid} (sc={ss[pid]:.2f})')
ax.axhline(1, color='black', linestyle='--', linewidth=0.7, label='|C_α|=1 (sufficiency)')
for k in [3, 5, 7]:
    ax.axvline(k, color='grey', linestyle=':', linewidth=0.7)
ax.set_xlabel('cumulative valid nights (k)')
ax.set_ylabel('|C_α|')
ax.set_title('Individual sufficiency trajectories (synthetic, α=0.10)')
ax.legend(loc='upper right', fontsize=7)
ax.set_yticks([1, 2])
Path('docs/figures').mkdir(parents=True, exist_ok=True)
out = Path('docs/figures/student3_m3_sufficiency_curves.png')
fig.tight_layout(); fig.savefig(out, dpi=150); plt.show()
print(f'saved {out}')


## 8. Summary


In [ ]:
conv = combined[combined['strategy_name'] == 'covariate_conditional']
n_converged = (conv['convergence_status'] == 'converged').sum()
tau_values = conv.loc[conv['convergence_status'] == 'converged', 'tau_i'].astype(int)
print(f'Covariate-conditional: {n_converged}/{len(conv)} converged')
print(f'τᵢ (converged only): median={tau_values.median():.0f}, IQR=[{tau_values.quantile(0.25):.0f}, {tau_values.quantile(0.75):.0f}], range=[{tau_values.min()}, {tau_values.max()}]')
print(f'Non-convergers: {len(conv) - n_converged}')


## Next (Milestone 4 and onward)

Not covered here:
- **Mondrian conformal.** Compute q̂ per stratum (e.g., signal_completeness × cycle_regular → 4 strata). Record within-stratum sample sizes. Write to `prediction_sets.mondrian_stratum`.
- **Regression meta-model.** Predict τᵢ from covariates (pipeline contract §11 item 6).
- **`collect_more` branch.** Activate once `valid_nights.quality_flag` is populated in real data.
- **Compare against all baselines.** WP5 writes fixed-rule + oracle + signal-quality decisions; WP7 consumes all of them along with this file.
